In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

spark.sql("USE CATALOG crypto_pipeline")

DataFrame[]

In [0]:
# Databricks notebook source

# ==========================================================
# Silver Layer
# Notebook: 01_build_dim_coin
#
# Purpose:
# Build dim_coin using SCD Type 2
#
# ==========================================================

from pyspark.sql import functions as F
from pyspark.sql.types import *
from pyspark.sql.window import Window

CATALOG = "crypto_pipeline"

BRONZE_TABLE = f"{CATALOG}.bronze.raw_coin_market_data"
DIM_TABLE = f"{CATALOG}.silver.dim_coin"

spark.sql(f"USE CATALOG {CATALOG}")

# ==========================================================
# Create Dimension Table
# ==========================================================

spark.sql(f"""

CREATE TABLE IF NOT EXISTS {DIM_TABLE}

(

coin_sk STRING,

coin_id STRING,

name STRING,

symbol STRING,

image STRING,

record_hash STRING,

effective_start_date TIMESTAMP,

effective_end_date TIMESTAMP,

is_current BOOLEAN

)

USING DELTA

""")

# ==========================================================
# Read Bronze
# ==========================================================

bronze_df = spark.table(BRONZE_TABLE)

# ==========================================================
# Keep only latest record for each coin
# ==========================================================

window = Window.partitionBy("id").orderBy(F.col("_ingested_at").desc())

latest_df = (

    bronze_df

    .withColumn("rn", F.row_number().over(window))

    .filter("rn = 1")

    .drop("rn")

)

# ==========================================================
# Rename Columns
# ==========================================================

latest_df = (

    latest_df

    .withColumnRenamed("id", "coin_id")

)

# ==========================================================
# Deterministic Record Hash
# Used to detect attribute changes
# ==========================================================

latest_df = latest_df.withColumn(

    "record_hash",

    F.sha2(

        F.concat_ws(

            "||",

            F.col("coin_id"),

            F.col("name"),

            F.col("symbol"),

            F.col("image")

        ),

        256

    )

)

# ==========================================================
# Effective Start Date
# ==========================================================

latest_df = latest_df.withColumn(

    "effective_start_date",

    F.col("_ingested_at")

)

# ==========================================================
# Surrogate Key
#
# Deterministic:
# SHA256(coin_id + effective_start_date)
# ==========================================================

latest_df = latest_df.withColumn(

    "coin_sk",

    F.sha2(

        F.concat_ws(

            "||",

            F.col("coin_id"),

            F.col("effective_start_date").cast("string")

        ),

        256

    )

)

# ==========================================================
# Default Values
# ==========================================================

latest_df = (

    latest_df

    .withColumn(

        "effective_end_date",

        F.lit(None).cast("timestamp")

    )

    .withColumn(

        "is_current",

        F.lit(True)

    )

)

# ==========================================================
# Select Required Columns
# ==========================================================

staging_df = latest_df.select(

    "coin_sk",

    "coin_id",

    "name",

    "symbol",

    "image",

    "record_hash",

    "effective_start_date",

    "effective_end_date",

    "is_current"

)

print("Latest Bronze Records")

display(staging_df)

# ==========================================================
# Read Current Dimension
# ==========================================================

dim_df = spark.table(DIM_TABLE)

current_dim = dim_df.filter(

    F.col("is_current") == True

)

print("Current Dimension")

display(current_dim)

# ==========================================================
# Detect Brand New Coins
# ==========================================================

new_coins = (

    staging_df.alias("s")

    .join(

        current_dim.alias("d"),

        "coin_id",

        "leftanti"

    )

)

print("New Coins")

display(new_coins)

# ==========================================================
# Detect Changed Coins
# ==========================================================

changed = (

    staging_df.alias("s")

    .join(

        current_dim.alias("d"),

        "coin_id"

    )

    .filter(

        F.col("s.record_hash") !=

        F.col("d.record_hash")

    )

    .select("s.*")

)

print("Changed Coins")

display(changed)

# ==========================================================
# No writes happen in Part 1.
#
# Part 2 performs:
#
# 1. Expire old rows
# 2. Insert changed rows
# 3. Insert new rows
# 4. Validate SCD2
# ==========================================================

Latest Bronze Records


coin_sk,coin_id,name,symbol,image,record_hash,effective_start_date,effective_end_date,is_current
2e718161445afbd9a5ee5c5a22835a07defc2037fe5031de1fef450dd4aac94b,aster-2,Aster,aster,https://coin-images.coingecko.com/coins/images/69040/large/_ASTER.png?1757326782,661d88cd20fcb60ce50ee937112a849efc8830fdcff7187e25ac59d19ff8ec36,2026-07-22T08:59:21.134Z,null,true
4e2b4465ae2c4c917495b637780dff7ec2113e1f390b8aa84fbb19819b848768,avalanche-2,Avalanche,avax,https://coin-images.coingecko.com/coins/images/12559/large/Avalanche_Circle_RedWhite_Trans.png?1696512369,2fb0a9bbc491f9dc4443952dcd0825ee7bf6925946593956028cc566761c5194,2026-07-22T08:59:21.134Z,null,true
4be6b588bed71a3f34047d227bfad49e9f35d4a89195b7ad153f865a9450796a,binancecoin,BNB,bnb,https://coin-images.coingecko.com/coins/images/825/large/bnb-icon2_2x.png?1696501970,e98e87d8783235d1101d74975cc90861942d5a232ccb50387e332007d63ef988,2026-07-22T08:59:21.134Z,null,true
c18cad259c9f4f8ef98ef645b2aa3dac7d726ab041d06643d5d13ffff58d358c,bitcoin,Bitcoin,btc,https://coin-images.coingecko.com/coins/images/1/large/bitcoin.png?1696501400,4e6f63b502fe5775e95073e9932c848aa37224549f246039667c6aadf6842e42,2026-07-22T08:59:21.134Z,null,true
c4e03185c2e365e4f3bb253b58944a0252139e6cb7d6ecae7a7d2dde8ab47ef0,bitcoin-cash,Bitcoin Cash,bch,https://coin-images.coingecko.com/coins/images/780/large/bitcoin-cash-circle.png?1696501932,154225d3b43ac405397462f24a2e9336310a3fcaa979ec4ede2d6e50dee5df3b,2026-07-22T08:59:21.134Z,null,true
e44988b4e78b3b5f96854b7d376c9a1e5dddba8476196e5dd3000284a1488c5c,bittensor,Bittensor,tao,https://coin-images.coingecko.com/coins/images/28452/large/ARUsPeNQ_400x400.jpeg?1696527447,dbccc8d5bd22a363e819301dbe7432181c3fb31864f60f1fb94bc72e8eda825d,2026-07-22T08:59:21.134Z,null,true
e7735b4253b64a49c688af4708daa3dd47c829fd075ff74ea83fff1b99246733,blackrock-usd-institutional-digital-liquidity-fund,BlackRock USD Institutional Digital Liquidity Fund,buidl,https://coin-images.coingecko.com/coins/images/36291/large/blackrock.png?1711013223,c98c8ca6089b18e593242ae9aa567f155c4bec878b07576405938c5ea7122857,2026-07-22T08:59:21.134Z,null,true
126bda23e48f287091cc02ebdc4a972096a6bc774223da74dc40c1ab85f18249,canton-network,Canton,cc,https://coin-images.coingecko.com/coins/images/70468/large/Canton-Ticker_%281%29.png?1762826299,9fc024820f7be7766ab2263f3b1020504be637bac5486bdaba45f9f60cda0bcf,2026-07-22T08:59:21.134Z,null,true
6235bef37881100ed67526ae2a1af07ab4725e2e7b1f7ae709c29e01a6bc53e2,cardano,Cardano,ada,https://coin-images.coingecko.com/coins/images/975/large/cardano.png?1696502090,93343e16159bf7544f5b9ceae7e85e362f5176a7dae332687770251f6954733d,2026-07-22T08:59:21.134Z,null,true
b76527ae1b44a61572d49f1ac34393c9d4b2acc91346d9bdc978b09fe711be00,chainlink,Chainlink,link,https://coin-images.coingecko.com/coins/images/877/large/Chainlink_Logo_500.png?1760023405,942140fec66088cdfb799d7072a720e5b77995758fb0f784d4db676185b43408,2026-07-22T08:59:21.134Z,null,true


Current Dimension


coin_sk,coin_id,name,symbol,image,record_hash,effective_start_date,effective_end_date,is_current


New Coins


coin_id,coin_sk,name,symbol,image,record_hash,effective_start_date,effective_end_date,is_current
aster-2,2e718161445afbd9a5ee5c5a22835a07defc2037fe5031de1fef450dd4aac94b,Aster,aster,https://coin-images.coingecko.com/coins/images/69040/large/_ASTER.png?1757326782,661d88cd20fcb60ce50ee937112a849efc8830fdcff7187e25ac59d19ff8ec36,2026-07-22T08:59:21.134Z,null,true
avalanche-2,4e2b4465ae2c4c917495b637780dff7ec2113e1f390b8aa84fbb19819b848768,Avalanche,avax,https://coin-images.coingecko.com/coins/images/12559/large/Avalanche_Circle_RedWhite_Trans.png?1696512369,2fb0a9bbc491f9dc4443952dcd0825ee7bf6925946593956028cc566761c5194,2026-07-22T08:59:21.134Z,null,true
binancecoin,4be6b588bed71a3f34047d227bfad49e9f35d4a89195b7ad153f865a9450796a,BNB,bnb,https://coin-images.coingecko.com/coins/images/825/large/bnb-icon2_2x.png?1696501970,e98e87d8783235d1101d74975cc90861942d5a232ccb50387e332007d63ef988,2026-07-22T08:59:21.134Z,null,true
bitcoin,c18cad259c9f4f8ef98ef645b2aa3dac7d726ab041d06643d5d13ffff58d358c,Bitcoin,btc,https://coin-images.coingecko.com/coins/images/1/large/bitcoin.png?1696501400,4e6f63b502fe5775e95073e9932c848aa37224549f246039667c6aadf6842e42,2026-07-22T08:59:21.134Z,null,true
bitcoin-cash,c4e03185c2e365e4f3bb253b58944a0252139e6cb7d6ecae7a7d2dde8ab47ef0,Bitcoin Cash,bch,https://coin-images.coingecko.com/coins/images/780/large/bitcoin-cash-circle.png?1696501932,154225d3b43ac405397462f24a2e9336310a3fcaa979ec4ede2d6e50dee5df3b,2026-07-22T08:59:21.134Z,null,true
bittensor,e44988b4e78b3b5f96854b7d376c9a1e5dddba8476196e5dd3000284a1488c5c,Bittensor,tao,https://coin-images.coingecko.com/coins/images/28452/large/ARUsPeNQ_400x400.jpeg?1696527447,dbccc8d5bd22a363e819301dbe7432181c3fb31864f60f1fb94bc72e8eda825d,2026-07-22T08:59:21.134Z,null,true
blackrock-usd-institutional-digital-liquidity-fund,e7735b4253b64a49c688af4708daa3dd47c829fd075ff74ea83fff1b99246733,BlackRock USD Institutional Digital Liquidity Fund,buidl,https://coin-images.coingecko.com/coins/images/36291/large/blackrock.png?1711013223,c98c8ca6089b18e593242ae9aa567f155c4bec878b07576405938c5ea7122857,2026-07-22T08:59:21.134Z,null,true
canton-network,126bda23e48f287091cc02ebdc4a972096a6bc774223da74dc40c1ab85f18249,Canton,cc,https://coin-images.coingecko.com/coins/images/70468/large/Canton-Ticker_%281%29.png?1762826299,9fc024820f7be7766ab2263f3b1020504be637bac5486bdaba45f9f60cda0bcf,2026-07-22T08:59:21.134Z,null,true
cardano,6235bef37881100ed67526ae2a1af07ab4725e2e7b1f7ae709c29e01a6bc53e2,Cardano,ada,https://coin-images.coingecko.com/coins/images/975/large/cardano.png?1696502090,93343e16159bf7544f5b9ceae7e85e362f5176a7dae332687770251f6954733d,2026-07-22T08:59:21.134Z,null,true
chainlink,b76527ae1b44a61572d49f1ac34393c9d4b2acc91346d9bdc978b09fe711be00,Chainlink,link,https://coin-images.coingecko.com/coins/images/877/large/Chainlink_Logo_500.png?1760023405,942140fec66088cdfb799d7072a720e5b77995758fb0f784d4db676185b43408,2026-07-22T08:59:21.134Z,null,true


Changed Coins


coin_id,coin_sk,name,symbol,image,record_hash,effective_start_date,effective_end_date,is_current


In [0]:
# ==========================================================
# PART 2
# SCD TYPE 2 MERGE LOGIC
# ==========================================================

from delta.tables import DeltaTable

dim_delta = DeltaTable.forName(spark, DIM_TABLE)

# ==========================================================
# Expire Existing Current Records
# ==========================================================

if changed.count() > 0:

    expire_df = changed.select(
        "coin_id",
        "effective_start_date"
    )

    (
        dim_delta.alias("target")
        .merge(
            expire_df.alias("source"),
            """
            target.coin_id = source.coin_id
            AND target.is_current = true
            """
        )
        .whenMatchedUpdate(
            set={
                "effective_end_date": "source.effective_start_date",
                "is_current": "false"
            }
        )
        .execute()
    )

print("Existing rows expired.")

# ==========================================================
# Insert Changed Records
# ==========================================================

if changed.count() > 0:

    (
        changed.write
        .mode("append")
        .saveAsTable(DIM_TABLE)
    )

print("Changed records inserted.")

# ==========================================================
# Insert Brand New Coins
# ==========================================================

if new_coins.count() > 0:

    (
        new_coins.write
        .mode("append")
        .saveAsTable(DIM_TABLE)
    )

print("New coins inserted.")

# ==========================================================
# Final Dimension
# ==========================================================

final_dim = spark.table(DIM_TABLE)

display(
    final_dim.orderBy(
        "coin_id",
        "effective_start_date"
    )
)

print("Current row count:",
      final_dim.filter("is_current = true").count())

print("Total historical rows:",
      final_dim.count())

Existing rows expired.
Changed records inserted.
New coins inserted.


coin_sk,coin_id,name,symbol,image,record_hash,effective_start_date,effective_end_date,is_current
2e718161445afbd9a5ee5c5a22835a07defc2037fe5031de1fef450dd4aac94b,aster-2,Aster,aster,https://coin-images.coingecko.com/coins/images/69040/large/_ASTER.png?1757326782,661d88cd20fcb60ce50ee937112a849efc8830fdcff7187e25ac59d19ff8ec36,2026-07-22T08:59:21.134Z,null,true
4e2b4465ae2c4c917495b637780dff7ec2113e1f390b8aa84fbb19819b848768,avalanche-2,Avalanche,avax,https://coin-images.coingecko.com/coins/images/12559/large/Avalanche_Circle_RedWhite_Trans.png?1696512369,2fb0a9bbc491f9dc4443952dcd0825ee7bf6925946593956028cc566761c5194,2026-07-22T08:59:21.134Z,null,true
4be6b588bed71a3f34047d227bfad49e9f35d4a89195b7ad153f865a9450796a,binancecoin,BNB,bnb,https://coin-images.coingecko.com/coins/images/825/large/bnb-icon2_2x.png?1696501970,e98e87d8783235d1101d74975cc90861942d5a232ccb50387e332007d63ef988,2026-07-22T08:59:21.134Z,null,true
c18cad259c9f4f8ef98ef645b2aa3dac7d726ab041d06643d5d13ffff58d358c,bitcoin,Bitcoin,btc,https://coin-images.coingecko.com/coins/images/1/large/bitcoin.png?1696501400,4e6f63b502fe5775e95073e9932c848aa37224549f246039667c6aadf6842e42,2026-07-22T08:59:21.134Z,null,true
c4e03185c2e365e4f3bb253b58944a0252139e6cb7d6ecae7a7d2dde8ab47ef0,bitcoin-cash,Bitcoin Cash,bch,https://coin-images.coingecko.com/coins/images/780/large/bitcoin-cash-circle.png?1696501932,154225d3b43ac405397462f24a2e9336310a3fcaa979ec4ede2d6e50dee5df3b,2026-07-22T08:59:21.134Z,null,true
e44988b4e78b3b5f96854b7d376c9a1e5dddba8476196e5dd3000284a1488c5c,bittensor,Bittensor,tao,https://coin-images.coingecko.com/coins/images/28452/large/ARUsPeNQ_400x400.jpeg?1696527447,dbccc8d5bd22a363e819301dbe7432181c3fb31864f60f1fb94bc72e8eda825d,2026-07-22T08:59:21.134Z,null,true
e7735b4253b64a49c688af4708daa3dd47c829fd075ff74ea83fff1b99246733,blackrock-usd-institutional-digital-liquidity-fund,BlackRock USD Institutional Digital Liquidity Fund,buidl,https://coin-images.coingecko.com/coins/images/36291/large/blackrock.png?1711013223,c98c8ca6089b18e593242ae9aa567f155c4bec878b07576405938c5ea7122857,2026-07-22T08:59:21.134Z,null,true
126bda23e48f287091cc02ebdc4a972096a6bc774223da74dc40c1ab85f18249,canton-network,Canton,cc,https://coin-images.coingecko.com/coins/images/70468/large/Canton-Ticker_%281%29.png?1762826299,9fc024820f7be7766ab2263f3b1020504be637bac5486bdaba45f9f60cda0bcf,2026-07-22T08:59:21.134Z,null,true
6235bef37881100ed67526ae2a1af07ab4725e2e7b1f7ae709c29e01a6bc53e2,cardano,Cardano,ada,https://coin-images.coingecko.com/coins/images/975/large/cardano.png?1696502090,93343e16159bf7544f5b9ceae7e85e362f5176a7dae332687770251f6954733d,2026-07-22T08:59:21.134Z,null,true
b76527ae1b44a61572d49f1ac34393c9d4b2acc91346d9bdc978b09fe711be00,chainlink,Chainlink,link,https://coin-images.coingecko.com/coins/images/877/large/Chainlink_Logo_500.png?1760023405,942140fec66088cdfb799d7072a720e5b77995758fb0f784d4db676185b43408,2026-07-22T08:59:21.134Z,null,true


Current row count: 50
Total historical rows: 50


In [0]:
# ==========================================================
# VALIDATION
# ==========================================================

print("Checking for duplicate current rows...")

duplicates = (

    spark.table(DIM_TABLE)

    .filter("is_current = true")

    .groupBy("coin_id")

    .count()

    .filter("count > 1")

)

display(duplicates)

print("Duplicate current rows:",
      duplicates.count())

print("Checking history...")

display(

    spark.table(DIM_TABLE)

    .orderBy(

        "coin_id",

        "effective_start_date"

    )

)

print("SCD Type 2 build completed.")

Checking for duplicate current rows...


coin_id,count


Duplicate current rows: 0
Checking history...


coin_sk,coin_id,name,symbol,image,record_hash,effective_start_date,effective_end_date,is_current
2e718161445afbd9a5ee5c5a22835a07defc2037fe5031de1fef450dd4aac94b,aster-2,Aster,aster,https://coin-images.coingecko.com/coins/images/69040/large/_ASTER.png?1757326782,661d88cd20fcb60ce50ee937112a849efc8830fdcff7187e25ac59d19ff8ec36,2026-07-22T08:59:21.134Z,null,true
4e2b4465ae2c4c917495b637780dff7ec2113e1f390b8aa84fbb19819b848768,avalanche-2,Avalanche,avax,https://coin-images.coingecko.com/coins/images/12559/large/Avalanche_Circle_RedWhite_Trans.png?1696512369,2fb0a9bbc491f9dc4443952dcd0825ee7bf6925946593956028cc566761c5194,2026-07-22T08:59:21.134Z,null,true
4be6b588bed71a3f34047d227bfad49e9f35d4a89195b7ad153f865a9450796a,binancecoin,BNB,bnb,https://coin-images.coingecko.com/coins/images/825/large/bnb-icon2_2x.png?1696501970,e98e87d8783235d1101d74975cc90861942d5a232ccb50387e332007d63ef988,2026-07-22T08:59:21.134Z,null,true
c18cad259c9f4f8ef98ef645b2aa3dac7d726ab041d06643d5d13ffff58d358c,bitcoin,Bitcoin,btc,https://coin-images.coingecko.com/coins/images/1/large/bitcoin.png?1696501400,4e6f63b502fe5775e95073e9932c848aa37224549f246039667c6aadf6842e42,2026-07-22T08:59:21.134Z,null,true
c4e03185c2e365e4f3bb253b58944a0252139e6cb7d6ecae7a7d2dde8ab47ef0,bitcoin-cash,Bitcoin Cash,bch,https://coin-images.coingecko.com/coins/images/780/large/bitcoin-cash-circle.png?1696501932,154225d3b43ac405397462f24a2e9336310a3fcaa979ec4ede2d6e50dee5df3b,2026-07-22T08:59:21.134Z,null,true
e44988b4e78b3b5f96854b7d376c9a1e5dddba8476196e5dd3000284a1488c5c,bittensor,Bittensor,tao,https://coin-images.coingecko.com/coins/images/28452/large/ARUsPeNQ_400x400.jpeg?1696527447,dbccc8d5bd22a363e819301dbe7432181c3fb31864f60f1fb94bc72e8eda825d,2026-07-22T08:59:21.134Z,null,true
e7735b4253b64a49c688af4708daa3dd47c829fd075ff74ea83fff1b99246733,blackrock-usd-institutional-digital-liquidity-fund,BlackRock USD Institutional Digital Liquidity Fund,buidl,https://coin-images.coingecko.com/coins/images/36291/large/blackrock.png?1711013223,c98c8ca6089b18e593242ae9aa567f155c4bec878b07576405938c5ea7122857,2026-07-22T08:59:21.134Z,null,true
126bda23e48f287091cc02ebdc4a972096a6bc774223da74dc40c1ab85f18249,canton-network,Canton,cc,https://coin-images.coingecko.com/coins/images/70468/large/Canton-Ticker_%281%29.png?1762826299,9fc024820f7be7766ab2263f3b1020504be637bac5486bdaba45f9f60cda0bcf,2026-07-22T08:59:21.134Z,null,true
6235bef37881100ed67526ae2a1af07ab4725e2e7b1f7ae709c29e01a6bc53e2,cardano,Cardano,ada,https://coin-images.coingecko.com/coins/images/975/large/cardano.png?1696502090,93343e16159bf7544f5b9ceae7e85e362f5176a7dae332687770251f6954733d,2026-07-22T08:59:21.134Z,null,true
b76527ae1b44a61572d49f1ac34393c9d4b2acc91346d9bdc978b09fe711be00,chainlink,Chainlink,link,https://coin-images.coingecko.com/coins/images/877/large/Chainlink_Logo_500.png?1760023405,942140fec66088cdfb799d7072a720e5b77995758fb0f784d4db676185b43408,2026-07-22T08:59:21.134Z,null,true


SCD Type 2 build completed.
